# Demo notebook
This notebook demos our experiments with large language models. You will need to provide the api key as the `OPENAI_API_KEY` environment variable to execute this notebook the open-ai api based llms.

In [35]:
!pip install langchain openai chromadb unstructured evaluate bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 31.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.3/98.3 kB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.2 MB/s eta 0:00:00


In [1]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI
from langchain.document_loaders import TextLoader, DirectoryLoader
from langchain.chains.qa_with_sources import load_qa_with_sources_chain
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import os

In [2]:
load_dotenv(find_dotenv("credentials.env"), override=True)

True

## Load all the documents in a vectorstore

In [8]:
loader = DirectoryLoader('../data/external', glob="**/*.md", loader_cls=TextLoader)
documents = loader.load()

In [9]:
## To do: better split of docs
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()

In [10]:
docsearch = Chroma.from_documents(texts, embeddings, persist_directory='../data/interim')
docsearch.persist()

Running Chroma using direct local API.
loaded in 850 embeddings
loaded in 1 collections


/opt/anaconda3/envs/fm/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exiting: Cleaning up .chroma directory
Persisting DB to disk, putting it in the save folder ../data/interim


In [11]:
chain = load_qa_with_sources_chain(OpenAI(temperature=0), chain_type="stuff")

In [12]:
def answer_question(query, index, chain):
    """
    Takes in query, index to search from, and llm chain to generate answer
    """
    ## Retrieve docs
    docs = index.similarity_search(query)
    ## Generate answer
    answer = chain({"input_documents": docs, "question": query}, return_only_outputs=True)
    return answer['output_text']

# Evaluation using the FAQ dataset and metrics 

In [42]:
eval_dataset = pd.read_csv('../data/processed/validation_data.csv')[['Question', 'Answer']]

In [44]:
questions = eval_dataset['Question'].tolist()
real_answers = eval_dataset['Answer'].tolist()
generated_answers = list()

for query in eval_dataset['Question'].tolist():
    answer = answer_question(query, docsearch, chain)
    generated_answers.append(answer)
    
eval_dataset['generated_answers'] = generated_answers

In [50]:
examples = [{'question':questions[i],
             'answer':real_answers[i]} for i in range(len(questions))]
predictions = [{'text':generated_answers[i]} for i in range(len(generated_answers))]

In [55]:
from langchain.prompts.prompt import PromptTemplate
from langchain.evaluation.qa import QAEvalChain

_PROMPT_TEMPLATE = """You are an expert Openshift engineer specialized in answering user queries about ROSA.
You are grading the following question:
{query}
Here is the real answer:
{answer}
You are grading the following predicted answer:
{result}
What grade do you give from 0 to 100, where 0 is the lowest (very low similarity) and 100 is the highest (very high similarity)?
"""

PROMPT = PromptTemplate(input_variables=["query", "answer", "result"], template=_PROMPT_TEMPLATE)

In [56]:
evalchain = QAEvalChain.from_llm(llm=llm,prompt=PROMPT)
graded_outputs = evalchain.evaluate(examples, predictions, question_key="question", answer_key="answer", prediction_key="text")

In [57]:
eval_dataset['llm_grading'] = graded_outputs
eval_dataset

,Question,Answer,generated_answers,llm_grading
0,What is Red Hat OpenShift Service on AWS (ROSA)?,Red Hat Openshift Service on AWS (ROSA) is a f...,Red Hat OpenShift Service on AWS (ROSA) is a ...,{'text': ' 100'}
1,Where can I go to get more information/details?,- [ROSA Webpage](https://www.openshift.com/pro...,You can visit the Red Hat Customer Portal to ...,{'text': ' 80'}
2,What are the benefits of Red Hat OpenShift Ser...,- **Native AWS service:** Access and use Red H...,The benefits of Red Hat OpenShift Service on ...,{'text': ' 90'}
3,What are the differences between Red Hat OpenS...,Everything you need to deploy and manage conta...,Red Hat OpenShift Service on AWS includes add...,{'text': ' 90'}
4,What exactly am I responsible for and what is ...,"In short, anything that is related to deployin...",Anything that is related to deploying the clu...,{'text': ' 90'}
...,...,...,...,...
60,What features are upcoming for ROSA?,The current ROSA roadmap can be seen at: [http...,Features upcoming for ROSA can be found in th...,{'text': ' 50'}
61,What kind of instances are supported for worke...,See [AWS compute types](https://docs.openshift...,See [AWS compute types](https://docs.openshif...,{'text': ' 100'}
62,"Does ROSA support an air-gapped, disconnected ...","No, the ROSA cluster must have egress to the i...","No, the ROSA cluster must have egress to the ...",{'text': ' 90'}
63,Is node autoscaling available?,Yes. Autoscaling allows you to automatically a...,Node autoscaling is available.\nSOURCES: ../d...,{'text': ' 80'}


In [ ]:
# from evaluate import load
# bertscore = load("bertscore")

## To do

* Add bertcore, ROGUE, BLEU, WER